# 内科2天以内住院患者QA筛选

## 目标
基于已筛选的内科2天以内住院患者，进一步筛选适合生成用药管理QA的病例

## 筛选条件
1. 前病史不为空
2. 排除肿瘤/移植/手术
3. 入院后Vitals/Labs不为空
4. GT用药1-10种（持续用药 + 新增用药）
5. 人口学信息完整（年龄、性别）

## GT用药定义
- **持续用药**: 入院前30天 + 出院后30天都有的口服药
- **新增用药**: 仅在出院后30天有的口服药
- **GT = 持续用药 + 新增用药**
- 排除: IV药物、麻醉药、化疗药、免疫抑制剂

## 说明
- 目标：一般内科用药管理QA
- 不限制慢性病，包含所有有前病史的内科患者
- 排除肿瘤/移植/手术患者，确保GT用药是慢病管理药物

In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# 初始化统计字典
stats = {
    "total": 0,
    "no_precond": 0,
    "excluded_condition": 0,
    "no_vitals_labs": 0,
    "gt_range": 0,
    "no_demo": 0,
    "ok": 0,
}

print("导入完成，统计字典已初始化")

导入完成，统计字典已初始化


In [3]:
print("=" * 80)
print("步骤1: 加载数据")
print("=" * 80)

EHRSHOT_CSV       = "/data/ehr/EHRSHOT/EHRSHOT_ASSETS/data/ehrshot.csv"
DRUG_EXPOSURE_CSV = "/home/bingkun_zhao/ehrshot_analysis/data/drug_exposure.csv"
CONCEPT_CSV       = "/home/bingkun_zhao/ehrshot_analysis/data/concept.csv"
CONDITION_CSV     = "/home/bingkun_zhao/ehrshot_analysis/data/condition_occurrence.csv"

print(f"\n加载EHRSHOT数据: {EHRSHOT_CSV}")
df = pd.read_csv(EHRSHOT_CSV, low_memory=False)
print(f"总记录数: {len(df):,}")

print("\n分离OMOP表...")
person     = df[df["omop_table"] == "person"].copy()
conditions = df[df["omop_table"] == "condition_occurrence"].copy()
measures   = df[df["omop_table"] == "measurement"].copy()

# visit_occurrence 从本地 data/ 读取（含科室信息）
filtered_visits = pd.read_csv("/home/bingkun_zhao/ehrshot_analysis/output/internal_medicine_2days_visits.csv")
print(f"  person: {len(person):,}")
print(f"  conditions: {len(conditions):,}")
print(f"  measures: {len(measures):,}")
print(f"  filtered_visits: {len(filtered_visits):,} (内科≤2天住院)")

print(f"\n加载本地drug_exposure数据: {DRUG_EXPOSURE_CSV}")
drugs = pd.read_csv(DRUG_EXPOSURE_CSV, low_memory=False)
print(f"  drugs: {len(drugs):,}")

print(f"\n加载concept数据: {CONCEPT_CSV}")
concept = pd.read_csv(CONCEPT_CSV, low_memory=False)
concept_map = dict(zip(concept['concept_id'], concept['concept_name']))
print(f"  concept: {len(concept):,}")

print(f"\n加载本地condition_occurrence数据: {CONDITION_CSV}")
visit_conditions = pd.read_csv(CONDITION_CSV, low_memory=False)
print(f"  visit_conditions: {len(visit_conditions):,}")

# ── 预处理日期（只做一次）────────────────────────────────────────────────
print("\n预处理日期字段...")
conditions['start_dt'] = pd.to_datetime(conditions['start'], errors='coerce')
measures['start_dt']   = pd.to_datetime(measures['start'], errors='coerce')
drugs['start_dt']      = pd.to_datetime(drugs['drug_exposure_start_DATE'], errors='coerce')

# ── 按 person_id 建索引 ───────────────────────────────────────────────────
print("建立 person_id 索引...")
conditions_by_patient = {pid: grp for pid, grp in conditions.groupby('patient_id')}
measures_by_patient   = {pid: grp for pid, grp in measures.groupby('patient_id')}
drugs_by_patient      = {pid: grp for pid, grp in drugs.groupby('person_id')}
visit_cond_by_visit   = {vid: grp for vid, grp in visit_conditions.groupby('visit_occurrence_id')}

stats["total"] = len(filtered_visits)
print(f"\n初始化完成，总visits: {stats['total']}")
print(f"  conditions索引患者数: {len(conditions_by_patient)}")
print(f"  measures索引患者数:   {len(measures_by_patient)}")
print(f"  drugs索引患者数:      {len(drugs_by_patient)}")

步骤1: 加载数据

加载EHRSHOT数据: /data/ehr/EHRSHOT/EHRSHOT_ASSETS/data/ehrshot.csv
总记录数: 41,661,637

分离OMOP表...
  person: 24,928
  conditions: 1,219,611
  measures: 30,622,455
  filtered_visits: 1,405 (内科≤2天住院)

加载本地drug_exposure数据: /home/bingkun_zhao/ehrshot_analysis/data/drug_exposure.csv
  drugs: 2,583,444

加载concept数据: /home/bingkun_zhao/ehrshot_analysis/data/concept.csv
  concept: 8,263,683

加载本地condition_occurrence数据: /home/bingkun_zhao/ehrshot_analysis/data/condition_occurrence.csv
  visit_conditions: 2,570,607

预处理日期字段...
建立 person_id 索引...

初始化完成，总visits: 1405
  conditions索引患者数: 6144
  measures索引患者数:   6662
  drugs索引患者数:      5925


In [4]:
# 排除条件定义（与generate_qa_with_atc.py保持一致）

from medical_code_mapping import get_code_description, get_medication_name

# 排除的前病史/入院诊断关键词（肿瘤/移植/手术）
EXCLUDE_CONDITION_KEYWORDS = [
    "malignant neoplasm", "carcinoma", "malignant neoplastic",
    "metastatic malignant", "primary malignant", "cancer",
    "malignant neoplastic disease", "overlapping malignant",
    "transplanted", "transplantation", "transplant",
    "complication of surgical procedure", "postoperative",
]

# 排除的药物关键词（麻醉/化疗/免疫抑制剂）
EXCLUDE_DRUG_KEYWORDS = [
    'lidocaine', 'propofol', 'ketamine', 'sevoflurane', 'fentanyl',
    'rocuronium', 'vecuronium', 'succinylcholine', 'midazolam',
    'cisplatin', 'carboplatin', 'doxorubicin', 'paclitaxel',
    'tacrolimus', 'cyclosporine', 'sirolimus', 'mycophenolate',
]

# 口服药物剂型关键词
ORAL_FORM_KEYWORDS = [
    'oral tablet', 'oral capsule', 'oral solution', 'oral suspension',
    'chewable tablet', 'disintegrating oral tablet', 'extended release oral',
    'delayed release oral', 'oral powder',
]

def has_excluded_condition(condition_name: str) -> bool:
    name_lower = condition_name.lower()
    return any(kw in name_lower for kw in EXCLUDE_CONDITION_KEYWORDS)

def is_oral_medication(drug_name: str) -> bool:
    return any(kw in drug_name.lower() for kw in ORAL_FORM_KEYWORDS)

def is_excluded_medication(drug_name: str) -> bool:
    return any(kw in drug_name.lower() for kw in EXCLUDE_DRUG_KEYWORDS)

def get_med_name(concept_id):
    return concept_map.get(concept_id, str(concept_id))

print("排除条件已定义")
print(f"  排除前病史/入院诊断关键词: {len(EXCLUDE_CONDITION_KEYWORDS)} 个")
print(f"  排除药物关键词: {len(EXCLUDE_DRUG_KEYWORDS)} 个")
print(f"  口服药物剂型关键词: {len(ORAL_FORM_KEYWORDS)} 个")


排除条件已定义
  排除前病史/入院诊断关键词: 13 个
  排除药物关键词: 17 个
  口服药物剂型关键词: 9 个


## 步骤2: 定义筛选常量和函数

In [5]:
print("=" * 80)
print("筛选条件1: 前病史不为空")
print("=" * 80)

has_precond = []
for _, row in filtered_visits.iterrows():
    patient_id   = row["person_id"]
    admission_dt = pd.to_datetime(row["visit_start_date"])
    pat_cond = conditions_by_patient.get(patient_id)
    has_cond = (pat_cond is not None) and (pat_cond['start_dt'] < admission_dt).any()
    has_precond.append(has_cond)

filtered_visits['has_precond'] = has_precond
visits_with_precond = filtered_visits[filtered_visits['has_precond']].copy()
stats["no_precond"] = (~filtered_visits['has_precond']).sum()

print(f"\n筛选前: {stats['total']} visits")
print(f"筛选后: {len(visits_with_precond)} visits")
print(f"筛选掉: {stats['no_precond']} visits (无前病史)")
print(f"保留比例: {len(visits_with_precond)/stats['total']*100:.1f}%")

筛选条件1: 前病史不为空

筛选前: 1405 visits
筛选后: 1390 visits
筛选掉: 15 visits (无前病史)
保留比例: 98.9%


In [6]:
print("=" * 80)
print("筛选条件2: 排除肿瘤/移植/手术（前病史 + 入院诊断双重检查）")
print("=" * 80)

def check_excluded(patient_id, admission_dt, visit_id):
    admission_date_start = pd.Timestamp(admission_dt.date())

    # 检查前病史
    pat_cond = conditions_by_patient.get(patient_id)
    if pat_cond is not None:
        pre_cond = pat_cond[pat_cond['start_dt'] < admission_date_start]
        for _, cr in pre_cond.iterrows():
            desc = get_code_description(cr["code"])
            if has_excluded_condition(desc):
                return True

    # 检查入院诊断
    adm_dx = visit_cond_by_visit.get(visit_id)
    if adm_dx is not None:
        for _, dx in adm_dx.iterrows():
            dx_name = concept_map.get(dx["condition_concept_id"], "")
            if has_excluded_condition(dx_name):
                return True

    return False

print("检查前病史和入院诊断中的排除关键词...")
visits_with_precond["has_excluded"] = visits_with_precond.apply(
    lambda row: check_excluded(
        row["person_id"],
        pd.to_datetime(row["visit_start_date"]),
        row["visit_occurrence_id"]
    ),
    axis=1
)

visits_no_exclude = visits_with_precond[~visits_with_precond["has_excluded"]].copy()
stats["excluded_condition"] = visits_with_precond["has_excluded"].sum()

print(f"\n筛选前: {len(visits_with_precond)} visits")
print(f"筛选后: {len(visits_no_exclude)} visits")
print(f"筛选掉: {stats['excluded_condition']} visits (包含肿瘤/移植/手术)")
print(f"保留比例: {len(visits_no_exclude)/len(visits_with_precond)*100:.1f}%")

筛选条件2: 排除肿瘤/移植/手术（前病史 + 入院诊断双重检查）
检查前病史和入院诊断中的排除关键词...

筛选前: 1390 visits
筛选后: 504 visits
筛选掉: 886 visits (包含肿瘤/移植/手术)
保留比例: 36.3%


In [7]:
print("=" * 80)
print("筛选条件3: 入院后Vitals/Labs不为空")
print("=" * 80)

def has_vitals_labs(patient_id, admission_dt):
    admission_date_start = pd.Timestamp(admission_dt.date())
    cutoff_24h = admission_date_start + timedelta(hours=24)
    pat_m = measures_by_patient.get(patient_id)
    if pat_m is None:
        return False
    return ((pat_m['start_dt'] >= admission_date_start) & (pat_m['start_dt'] <= cutoff_24h)).any()

print("检查入院后Vitals/Labs...")
visits_no_exclude['has_measures'] = visits_no_exclude.apply(
    lambda row: has_vitals_labs(row["person_id"], pd.to_datetime(row["visit_start_date"])),
    axis=1
)

visits_with_measures = visits_no_exclude[visits_no_exclude['has_measures']].copy()
stats["no_vitals_labs"] = (~visits_no_exclude['has_measures']).sum()

print(f"\n筛选前: {len(visits_no_exclude)} visits")
print(f"筛选后: {len(visits_with_measures)} visits")
print(f"筛选掉: {stats['no_vitals_labs']} visits (无入院后Vitals/Labs)")
print(f"保留比例: {len(visits_with_measures)/len(visits_no_exclude)*100:.1f}%")

筛选条件3: 入院后Vitals/Labs不为空
检查入院后Vitals/Labs...

筛选前: 504 visits
筛选后: 502 visits
筛选掉: 2 visits (无入院后Vitals/Labs)
保留比例: 99.6%


In [8]:
print("=" * 80)
print("筛选条件4: GT用药1-10种（持续用药 + 新增用药）")
print("=" * 80)

PRE_ADMISSION_WINDOW_DAYS  = 30  # 入院前30天
POST_DISCHARGE_WINDOW_DAYS = 7   # 出院后7天

def count_gt_drugs(visit_id, patient_id, admission_dt, discharge_dt):
    """统计GT用药数量：持续用药 + 新增用药"""
    pat_drugs = drugs_by_patient.get(patient_id)
    if pat_drugs is None:
        return 0, 0, 0

    pre_window_start = admission_dt - timedelta(days=PRE_ADMISSION_WINDOW_DAYS)
    post_window_end  = discharge_dt + timedelta(days=POST_DISCHARGE_WINDOW_DAYS)

    # 入院前30天的口服药
    pre_mask = (pat_drugs['start_dt'] >= pre_window_start) & (pat_drugs['start_dt'] < admission_dt)
    pre_oral_drugs = set()
    for _, dr in pat_drugs[pre_mask].iterrows():
        drug_name = concept_map.get(dr["drug_concept_id"], str(dr["drug_concept_id"]))
        if drug_name == str(dr["drug_concept_id"]):
            continue
        if not is_oral_medication(drug_name) or is_excluded_medication(drug_name):
            continue
        pre_oral_drugs.add(drug_name.split()[0].lower())

    # 出院后7天的口服药（排除本次visit）
    post_mask = (
        (pat_drugs['visit_occurrence_id'] != visit_id) &
        (pat_drugs['start_dt'] >= discharge_dt) &
        (pat_drugs['start_dt'] <= post_window_end)
    )
    post_oral_drugs = set()
    for _, dr in pat_drugs[post_mask].iterrows():
        drug_name = concept_map.get(dr["drug_concept_id"], str(dr["drug_concept_id"]))
        if drug_name == str(dr["drug_concept_id"]):
            continue
        if not is_oral_medication(drug_name) or is_excluded_medication(drug_name):
            continue
        post_oral_drugs.add(drug_name.split()[0].lower())

    continuous = pre_oral_drugs & post_oral_drugs
    new        = post_oral_drugs - pre_oral_drugs
    gt         = continuous | new
    return len(gt), len(continuous), len(new)

print(f"\n检查GT用药数量（入院前30天 + 出院后{POST_DISCHARGE_WINDOW_DAYS}天）...")
gt_results = visits_with_measures.apply(
    lambda row: count_gt_drugs(
        row['visit_occurrence_id'],
        row['person_id'],
        pd.to_datetime(row['visit_start_date']),
        pd.to_datetime(row['visit_end_date'])
    ),
    axis=1
)

visits_with_measures['gt_drug_count']       = gt_results.apply(lambda x: x[0])
visits_with_measures['gt_continuous_count'] = gt_results.apply(lambda x: x[1])
visits_with_measures['gt_new_count']        = gt_results.apply(lambda x: x[2])

visits_with_gt = visits_with_measures[
    (visits_with_measures['gt_drug_count'] >= 1) &
    (visits_with_measures['gt_drug_count'] <= 10)
].copy()

stats["gt_range"] = (
    (visits_with_measures['gt_drug_count'] < 1) |
    (visits_with_measures['gt_drug_count'] > 10)
).sum()

print(f"\n筛选前: {len(visits_with_measures)} visits")
print(f"筛选后: {len(visits_with_gt)} visits")
print(f"筛选掉: {stats['gt_range']} visits (GT用药不在1-10种范围)")

if len(visits_with_gt) > 0:
    print(f"\nGT用药类型分布:")
    print(f"  平均持续用药: {visits_with_gt['gt_continuous_count'].mean():.1f}种")
    print(f"  平均新增用药: {visits_with_gt['gt_new_count'].mean():.1f}种")
    print(f"  平均总用药:   {visits_with_gt['gt_drug_count'].mean():.1f}种")
    print(f"\nGT用药数量分布:")
    print(visits_with_gt['gt_drug_count'].value_counts().sort_index())
else:
    print("\n⚠ 没有visits通过筛选")

筛选条件4: GT用药1-10种（持续用药 + 新增用药）

检查GT用药数量（入院前30天 + 出院后7天）...



筛选前: 502 visits
筛选后: 236 visits
筛选掉: 266 visits (GT用药不在1-10种范围)

GT用药类型分布:
  平均持续用药: 3.3种
  平均新增用药: 3.4种
  平均总用药:   6.7种

GT用药数量分布:
1     12
2     12
3     11
4     15
5     25
6     17
7     31
8     37
9     40
10    36
Name: gt_drug_count, dtype: int64


In [11]:
print("=" * 80)
print("筛选条件5: 人口学信息完整（年龄、性别）")
print("=" * 80)

# 从 person 表提取 gender 和 birth_date，计算 age
gender_rows = person[person["code"].isin(["Gender/M", "Gender/F"])].copy()
gender_map = gender_rows.groupby("patient_id")["code"].first().map({"Gender/M": "Male", "Gender/F": "Female"})

birth_rows = person[person["code"] == "SNOMED/3950001"].copy()
birth_map = pd.to_datetime(birth_rows.groupby("patient_id")["start"].first(), errors="coerce")

# join 到 visits_with_gt
visits_with_gt["gender"] = visits_with_gt["person_id"].map(gender_map)
visits_with_gt["birth_date"] = visits_with_gt["person_id"].map(birth_map)
visits_with_gt["age"] = (
    (pd.to_datetime(visits_with_gt["visit_start_date"]) - visits_with_gt["birth_date"]).dt.days / 365.25
).round().astype("Int64")

has_demo = visits_with_gt["age"].notna() & visits_with_gt["gender"].notna()
final_visits = visits_with_gt[has_demo].copy()
stats["no_demo"] = (~has_demo).sum()
stats["ok"] = len(final_visits)

print(f"\n筛选前: {len(visits_with_gt)} visits")
print(f"筛选后: {len(final_visits)} visits")
print(f"筛选掉: {stats['no_demo']} visits (人口学信息不完整)")
print(f"保留比例: {len(final_visits)/len(visits_with_gt)*100:.1f}%")
print(f"\n年龄范围: {final_visits['age'].min()} - {final_visits['age'].max()} 岁")
print(f"性别分布: {final_visits['gender'].value_counts().to_dict()}")


筛选条件5: 人口学信息完整（年龄、性别）

筛选前: 236 visits
筛选后: 236 visits
筛选掉: 0 visits (人口学信息不完整)
保留比例: 100.0%

年龄范围: 19 - 87 岁
性别分布: {'Male': 149, 'Female': 87}


In [12]:
print("=" * 80)
print("筛选结果总结")
print("=" * 80)

summary_data = {
    "筛选步骤": [
        "初始visits",
        "1. 前病史不为空",
        "2. 排除肿瘤/移植/手术",
        "3. 入院后Vitals/Labs不为空",
        "4. GT用药1-10种",
        "5. 人口学信息完整",
    ],
    "剩余visits": [
        stats["total"],
        len(visits_with_precond),
        len(visits_no_exclude),
        len(visits_with_measures),
        len(visits_with_gt),
        len(final_visits),
    ],
    "筛选掉": [
        0,
        stats["no_precond"],
        stats["excluded_condition"],
        stats["no_vitals_labs"],
        stats["gt_range"],
        stats["no_demo"],
    ],
}

summary_df = pd.DataFrame(summary_data)
summary_df["保留比例(%)"] = (summary_df["剩余visits"] / stats["total"] * 100).round(1)
print("\n")
print(summary_df.to_string(index=False))

print(f"\n" + "=" * 80)
print("最终结果")
print("=" * 80)
print(f"初始visits: {stats['total']}")
print(f"最终通过: {len(final_visits)} ({len(final_visits)/stats['total']*100:.1f}%)")
print(f"总筛选掉: {stats['total'] - len(final_visits)} ({(stats['total'] - len(final_visits))/stats['total']*100:.1f}%)")

output_file = "/home/bingkun_zhao/ehrshot_analysis/output/qa_candidate/qa_candidate_visits.csv"
final_visits.to_csv(output_file, index=False)
print(f"\n✓ 已保存候选visits: {output_file}")
print(f"  记录数: {len(final_visits):,}")
print(f"  患者数: {final_visits['person_id'].nunique():,}")

print(f"\nGT用药统计:")
print(f"  平均持续用药: {final_visits['gt_continuous_count'].mean():.1f}种")
print(f"  平均新增用药: {final_visits['gt_new_count'].mean():.1f}种")
print(f"  平均总用药: {final_visits['gt_drug_count'].mean():.1f}种")


筛选结果总结


                筛选步骤  剩余visits  筛选掉  保留比例(%)
            初始visits      1405    0    100.0
           1. 前病史不为空      1390   15     98.9
       2. 排除肿瘤/移植/手术       504  886     35.9
3. 入院后Vitals/Labs不为空       502    2     35.7
        4. GT用药1-10种       236  266     16.8
          5. 人口学信息完整       236    0     16.8

最终结果
初始visits: 1405
最终通过: 236 (16.8%)
总筛选掉: 1169 (83.2%)

✓ 已保存候选visits: /home/bingkun_zhao/ehrshot_analysis/output/qa_candidate/qa_candidate_visits.csv
  记录数: 236
  患者数: 175

GT用药统计:
  平均持续用药: 3.3种
  平均新增用药: 3.4种
  平均总用药: 6.7种
